# Chapter 11 &mdash; The Dyck Language and Generalized Bracketing

**Concept 1 of the Chapter 11 decomposition:** *The Dyck Language and Generalized Bracketing*

Balanced parentheses with the prefix condition &mdash; the canonical context-free language.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-The-Dyck-Language/Concept-The-Dyck-Language.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$L_{Dyck}$ is the set of **balanced** parenthesis strings. Two conditions, and the
second is the one people forget:

1. the total number of `(` equals the total number of `)`;
2. **every prefix** has at least as many `(` as `)`.

Condition 2 rules out `)(`, which passes condition 1. Together they say: you may never
close a bracket you have not opened.

The same shape recurs everywhere &mdash; XML tags, `begin`/`end`, function calls and
returns, matched quotes. "Generalized bracketing" is the reason this one language
carries so much weight.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### The grammar, and the two-condition specification

In [ ]:
Dyck = mkg({'S': ["", "(S)", "SS"]})
show(Dyck)

def balanced(s):
    depth = 0
    for ch in s:
        depth += 1 if ch == '(' else -1
        if depth < 0: return False          # the PREFIX condition
    return depth == 0                        # the COUNT condition

## 3. Tests

The grammar generates exactly the balanced strings.

In [ ]:
L = language(Dyck, 8)
print("generated, up to length 8 :", L[:12], "...")
assert all(balanced(s) for s in L)
from itertools import product
allstr = [''.join(p) for k in range(0, 9, 2) for p in product('()', repeat=k)]
assert set(L) == {s for s in allstr if balanced(s)}
print("exactly the %d balanced strings of length <= 8" % len(L))

**The prefix condition is the subtle half.** `)(` has equal counts and is rejected.

In [ ]:
for s in [')(', '())(', '()', '(())', '()()', '(()']:
    print("  %-6r equal counts %-6s balanced %-6s in L(G) %s"
          % (s, s.count('(') == s.count(')'), balanced(s), s in L))
assert not balanced(')(') and ')(' in [x for x in allstr if x.count('(') == x.count(')')]

Nesting is unbounded, which is the whole point.

In [ ]:
for n in [1, 3, 6, 10]:
    s = '(' * n + ')' * n
    print("  depth %-3d %-24r balanced? %s" % (n, s if n <= 6 else '...', balanced(s)))
assert balanced('(' * 50 + ')' * 50)
print("\nNo finite-state machine can track an unbounded depth (Concept 2).")

The same shape, other brackets: XML-like tags.

In [ ]:
Tags = mkg({'S': ["", "aSb", "SS"]})
LT = language(Tags, 6)
print("with a = <t> and b = </t> :", LT)
assert set(LT) == {s.replace('(', 'a').replace(')', 'b') for s in language(Dyck, 6)}
print("\nIt is the same language with the symbols renamed -- generalized bracketing.")

## 4. Exercises


1. Write the two conditions as a single running-count invariant.
2. Give the Dyck language over **two** bracket kinds. Is `([)]` in it?
3. Which real file formats are Dyck-shaped? Which only look it?

In [ ]:
# Your work for the exercises above.